In [2]:
import pandas as pd
import numpy as np

reduced_dataset = pd.read_csv(
    "../data/processed/keyboard_reduced_productivity_dataset.csv"
)

print("Shape:", reduced_dataset.shape)
print("\nColumns:")
print(reduced_dataset.columns.tolist())

Shape: (10000, 10)

Columns:
['participant', 'session', 'window_id', 'DU.key1.key1_median', 'DU.key1.key2_median', 'unique_key1_count', 'unique_transition_count', 'consecutive_key1_repetition_rate', 'backspace_rate', 'productivity_score']


In [3]:
reduced_features = [
    "DU.key1.key1_median",
    "DU.key1.key2_median",
    "unique_key1_count",
    "unique_transition_count",
    "consecutive_key1_repetition_rate",
    "backspace_rate"
]

X = reduced_dataset[reduced_features]
y = reduced_dataset["productivity_score"]

groups = (
    reduced_dataset["participant"].astype(str)
    + "_session_"
    + reduced_dataset["session"].astype(str)
)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Number of features:", len(reduced_features))
print("Unique participant-session groups:", groups.nunique())

X shape: (10000, 6)
y shape: (10000,)
Number of features: 6
Unique participant-session groups: 197


In [4]:
from sklearn.model_selection import GroupKFold

group_kfold = GroupKFold(n_splits=5)

print("Number of folds:", group_kfold.get_n_splits())

Number of folds: 5


In [5]:
for fold, (train_idx, val_idx) in enumerate(
    group_kfold.split(X, y, groups=groups),
    start=1
):
    train_groups = groups.iloc[train_idx]
    val_groups = groups.iloc[val_idx]

    overlap = set(train_groups).intersection(set(val_groups))

    print(f"Fold {fold}")
    print("  Training samples:", len(train_idx))
    print("  Validation samples:", len(val_idx))
    print("  Training groups:", train_groups.nunique())
    print("  Validation groups:", val_groups.nunique())
    print("  Group overlap:", len(overlap))
    print()

Fold 1
  Training samples: 8007
  Validation samples: 1993
  Training groups: 158
  Validation groups: 39
  Group overlap: 0

Fold 2
  Training samples: 8019
  Validation samples: 1981
  Training groups: 158
  Validation groups: 39
  Group overlap: 0

Fold 3
  Training samples: 7977
  Validation samples: 2023
  Training groups: 157
  Validation groups: 40
  Group overlap: 0

Fold 4
  Training samples: 8020
  Validation samples: 1980
  Training groups: 158
  Validation groups: 39
  Group overlap: 0

Fold 5
  Training samples: 7977
  Validation samples: 2023
  Training groups: 157
  Validation groups: 40
  Group overlap: 0



In [6]:
%pip install xgboost

Note: you may need to restart the kernel to use updated packages.


In [7]:
import xgboost as xgb

print("XGBoost version:", xgb.__version__)

XGBoost version: 3.4.1


In [8]:
from xgboost import XGBRegressor

xgb_params = {
    "n_estimators": 300,
    "max_depth": 6,
    "learning_rate": 0.05,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "objective": "reg:squarederror",
    "random_state": 42,
    "n_jobs": -1
}

print("XGBoost parameters:")
for key, value in xgb_params.items():
    print(f"{key}: {value}")

XGBoost parameters:
n_estimators: 300
max_depth: 6
learning_rate: 0.05
subsample: 0.8
colsample_bytree: 0.8
objective: reg:squarederror
random_state: 42
n_jobs: -1


In [9]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

fold_results = []

for fold, (train_idx, val_idx) in enumerate(
    group_kfold.split(X, y, groups=groups),
    start=1
):
    print(f"Training Fold {fold}...")

    # Split data
    X_train_fold = X.iloc[train_idx]
    X_val_fold = X.iloc[val_idx]

    y_train_fold = y.iloc[train_idx]
    y_val_fold = y.iloc[val_idx]

    # Create a fresh model for this fold
    xgb_model = XGBRegressor(**xgb_params)

    # Train
    xgb_model.fit(
        X_train_fold,
        y_train_fold
    )

    # Predict
    y_pred_fold = xgb_model.predict(X_val_fold)

    # Metrics
    mae = mean_absolute_error(y_val_fold, y_pred_fold)
    mse = mean_squared_error(y_val_fold, y_pred_fold)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_val_fold, y_pred_fold)

    fold_results.append({
        "fold": fold,
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
        "R2": r2
    })

    print(f"  MAE : {mae:.4f}")
    print(f"  MSE : {mse:.4f}")
    print(f"  RMSE: {rmse:.4f}")
    print(f"  R²  : {r2:.4f}")
    print()

Training Fold 1...
  MAE : 0.4965
  MSE : 0.4466
  RMSE: 0.6683
  R²  : 0.9981

Training Fold 2...
  MAE : 0.5066
  MSE : 0.4435
  RMSE: 0.6659
  R²  : 0.9980

Training Fold 3...
  MAE : 0.5056
  MSE : 0.4317
  RMSE: 0.6570
  R²  : 0.9979

Training Fold 4...
  MAE : 0.5294
  MSE : 0.5058
  RMSE: 0.7112
  R²  : 0.9980

Training Fold 5...
  MAE : 0.5017
  MSE : 0.4420
  RMSE: 0.6648
  R²  : 0.9980



In [10]:
fold_results_df = pd.DataFrame(fold_results)

print("XGBoost 5-Fold Results")
print("----------------------")
print(fold_results_df)

print("\nMean ± Standard Deviation")
print("-------------------------")

for metric in ["MAE", "MSE", "RMSE", "R2"]:
    mean_value = fold_results_df[metric].mean()
    std_value = fold_results_df[metric].std()

    print(f"{metric}: {mean_value:.4f} ± {std_value:.4f}")

XGBoost 5-Fold Results
----------------------
   fold       MAE       MSE      RMSE        R2
0     1  0.496549  0.446621  0.668297  0.998052
1     2  0.506579  0.443458  0.665927  0.998045
2     3  0.505569  0.431687  0.657029  0.997921
3     4  0.529435  0.505803  0.711198  0.997999
4     5  0.501728  0.441996  0.664828  0.998014

Mean ± Standard Deviation
-------------------------
MAE: 0.5080 ± 0.0126
MSE: 0.4539 ± 0.0295
RMSE: 0.6735 ± 0.0215
R2: 0.9980 ± 0.0001


In [11]:
# Train final XGBoost model on the complete reduced dataset
final_xgb_model = XGBRegressor(**xgb_params)

final_xgb_model.fit(
    X,
    y
)

print("Final XGBoost model trained successfully.")

Final XGBoost model trained successfully.


In [12]:
xgb_importance = pd.DataFrame({
    "feature": reduced_features,
    "importance": final_xgb_model.feature_importances_
})

xgb_importance = xgb_importance.sort_values(
    "importance",
    ascending=False
).reset_index(drop=True)

print("XGBoost Feature Importance")
print("--------------------------")
print(xgb_importance)

print("\nImportance sum:", xgb_importance["importance"].sum())

XGBoost Feature Importance
--------------------------
                            feature  importance
0  consecutive_key1_repetition_rate    0.347606
1                    backspace_rate    0.220193
2                 unique_key1_count    0.132842
3           unique_transition_count    0.115663
4               DU.key1.key1_median    0.106103
5               DU.key1.key2_median    0.077593

Importance sum: 1.0000001


In [13]:
# Save XGBoost feature importance
xgb_importance.to_csv(
    "../data/processed/xgboost_feature_importance.csv",
    index=False
)

# Convert fold results list to DataFrame
xgb_fold_results = pd.DataFrame(fold_results)

# Save 5-fold cross-validation results
xgb_fold_results.to_csv(
    "../data/processed/xgboost_fold_results.csv",
    index=False
)

print("XGBoost results saved successfully.")

XGBoost results saved successfully.


In [14]:
print("XGBoost 5-Fold GroupKFold Summary")
print("----------------------------------")

print(f"MAE  : {xgb_fold_results['MAE'].mean():.4f} ± {xgb_fold_results['MAE'].std():.4f}")
print(f"MSE  : {xgb_fold_results['MSE'].mean():.4f} ± {xgb_fold_results['MSE'].std():.4f}")
print(f"RMSE : {xgb_fold_results['RMSE'].mean():.4f} ± {xgb_fold_results['RMSE'].std():.4f}")
print(f"R²   : {xgb_fold_results['R2'].mean():.4f} ± {xgb_fold_results['R2'].std():.4f}")

XGBoost 5-Fold GroupKFold Summary
----------------------------------
MAE  : 0.5080 ± 0.0126
MSE  : 0.4539 ± 0.0295
RMSE : 0.6735 ± 0.0215
R²   : 0.9980 ± 0.0001


In [15]:
# Load Random Forest and XGBoost feature importance
rf_importance = pd.read_csv(
    "../data/processed/random_forest_feature_importance.csv"
)

xgb_importance = pd.read_csv(
    "../data/processed/xgboost_feature_importance.csv"
)

# Rename importance columns
rf_importance = rf_importance.rename(
    columns={"importance": "random_forest_importance"}
)

xgb_importance = xgb_importance.rename(
    columns={"importance": "xgboost_importance"}
)

# Combine both models
importance_comparison = pd.merge(
    rf_importance,
    xgb_importance,
    on="feature"
)

# Sort by XGBoost importance
importance_comparison = importance_comparison.sort_values(
    "xgboost_importance",
    ascending=False
).reset_index(drop=True)

print("Random Forest vs XGBoost Feature Importance")
print("--------------------------------------------")
print(importance_comparison)

Random Forest vs XGBoost Feature Importance
--------------------------------------------
                            feature  random_forest_importance  \
0  consecutive_key1_repetition_rate                  0.412063   
1                    backspace_rate                  0.109670   
2                 unique_key1_count                  0.149125   
3           unique_transition_count                  0.078575   
4               DU.key1.key1_median                  0.159486   
5               DU.key1.key2_median                  0.091081   

   xgboost_importance  
0            0.347606  
1            0.220193  
2            0.132842  
3            0.115663  
4            0.106103  
5            0.077593  


In [16]:
# Save Random Forest vs XGBoost feature importance comparison

importance_comparison.to_csv(
    "../data/processed/rf_vs_xgboost_feature_importance.csv",
    index=False
)

print("Feature importance comparison saved successfully.")

Feature importance comparison saved successfully.


In [19]:
xgb_model = XGBRegressor(...)

c:\Users\DELL\OneDrive\Desktop\EffiSense-Keyboard-Productivity\venv\Lib\site-packages\xgboost\core.py:566: FutureWarning: Pass `objective` as keyword args.
  warnings.warn(msg, FutureWarning)


In [20]:

%whos

Variable                Type            Data/Info
-------------------------------------------------
GroupKFold              ABCMeta         <class 'sklearn.model_sel<...>ction._split.GroupKFold'>
Path                    type            <class 'pathlib._local.Path'>
X                       DataFrame       Shape: (10000, 6)
XGBRegressor            type            <class 'xgboost.sklearn.XGBRegressor'>
X_train_fold            DataFrame       Shape: (7977, 6)
X_val_fold              DataFrame       Shape: (2023, 6)
final_xgb_model         XGBRegressor    XGBRegressor(base_score=N<...>_parallel_tree=None, ...)
fold                    int             5
fold_results            list            n=5
fold_results_df         DataFrame       Shape: (5, 5)
group_kfold             GroupKFold      GroupKFold(n_splits=5, ra<...>tate=None, shuffle=False)
groups                  Series          Shape: (10000,)
importance_comparison   DataFrame       Shape: (6, 3)
joblib                  module          <

In [21]:

import joblib
from pathlib import Path

model_dir = Path("../models")
model_dir.mkdir(parents=True, exist_ok=True)

model_path = model_dir / "xgboost_keyboard_productivity.pkl"

joblib.dump(final_xgb_model, model_path)

print("XGBoost model saved successfully!")
print("Saved at:", model_path.resolve())

XGBoost model saved successfully!
Saved at: C:\Users\DELL\OneDrive\Desktop\EffiSense-Keyboard-Productivity\models\xgboost_keyboard_productivity.pkl


In [22]:

from xgboost import XGBRegressor
import joblib
from pathlib import Path

# Train a fresh model using the full feature dataset
deployment_model = XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

deployment_model.fit(X, y)

# Save the deployment model
model_dir = Path("../models")
model_dir.mkdir(parents=True, exist_ok=True)

model_path = model_dir / "xgboost_keyboard_productivity.pkl"
joblib.dump(deployment_model, model_path)

print("Deployment model trained and saved!")
print("Model path:", model_path.resolve())
print("Training rows:", len(X))
print("Features:", list(X.columns))

Deployment model trained and saved!
Model path: C:\Users\DELL\OneDrive\Desktop\EffiSense-Keyboard-Productivity\models\xgboost_keyboard_productivity.pkl
Training rows: 10000
Features: ['DU.key1.key1_median', 'DU.key1.key2_median', 'unique_key1_count', 'unique_transition_count', 'consecutive_key1_repetition_rate', 'backspace_rate']


In [23]:

import joblib
from pathlib import Path
import pandas as pd

# Load the saved model
model_path = Path("../models/xgboost_keyboard_productivity.pkl")
loaded_model = joblib.load(model_path)

# Select one example row
sample_input = X.iloc[[0]]

# Predict the keyboard productivity score
predicted_score = loaded_model.predict(sample_input)[0]

print("Input features:")
display(sample_input)

print(f"Predicted keyboard productivity score: {predicted_score:.2f}")

Input features:


,DU.key1.key1_median,DU.key1.key2_median,unique_key1_count,unique_transition_count,consecutive_key1_repetition_rate,backspace_rate
0,0.095,0.3045,20,38,0.0,0.0


Predicted keyboard productivity score: 63.79
